# Lab 04 — Primeiro Modelo de Classificação com Scikit-learn

> **Objetivo do lab:** treinar um primeiro modelo de classificação para prever cancelamento de clientes.

No Lab 03, fizemos uma análise exploratória do dataset de **Customer Churn**.

Agora vamos usar a base limpa para treinar um modelo simples de Machine Learning.

## Problema

Queremos responder:

> Com base nas características de um cliente, conseguimos prever se ele vai cancelar o serviço?

A variável alvo é:

```text
Churn
```

Ela possui duas classes:

- `No`: cliente não cancelou;
- `Yes`: cliente cancelou.

## O que vamos praticar

Neste lab, você vai aprender:

- separar variável alvo e variáveis explicativas;
- remover colunas que não devem entrar no modelo;
- transformar categorias em números com `pd.get_dummies()`;
- separar dados de treino e teste;
- treinar um modelo de classificação;
- fazer previsões;
- avaliar o modelo com acurácia;
- interpretar matriz de confusão;
- ler um `classification_report`.

**Importante:** este é o primeiro modelo.  
Ele não precisa ser perfeito. O foco é entender o fluxo.

## Como este lab funciona

Este notebook é **mais guiado que o Lab 03**, porque modelagem tem mais etapas novas.

Mas ainda existem partes para você completar.

Quando aparecer:

```python
resposta = None
```

substitua o `None` pelo código correto.

Use os comentários como guia.

# Parte 1 — Importar bibliotecas

Vamos usar:

- **Pandas** para manipular os dados;
- **Matplotlib** para gráficos simples;
- **Scikit-learn** para modelagem.

Neste lab, o modelo inicial será uma **Árvore de Decisão**.

Árvores de decisão são boas para começar porque são intuitivas:

> o modelo aprende regras do tipo: se contrato é mensal, se cobrança é alta, se cliente tem pouco tempo de contrato etc.

In [ ]:
# 1. Importe as bibliotecas principais.

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib

In [ ]:
# Teste do setup

assert "pd" in globals(), "Pandas precisa ser importado como pd."
assert "plt" in globals(), "Matplotlib precisa ser importado como plt."
assert "train_test_split" in globals(), "Importe train_test_split."
assert "DecisionTreeClassifier" in globals(), "Importe DecisionTreeClassifier."
assert "accuracy_score" in globals(), "Importe accuracy_score."
assert "joblib" in globals(), "Importe joblib."

print("Setup correto!")

# Parte 2 — Carregar a base limpa

Agora vamos carregar a versão limpa do dataset.

Essa base já passou pela limpeza mínima do Lab 03:

- `TotalCharges` foi convertido para número;
- linhas com `TotalCharges` inválido foram removidas.

In [ ]:
# URL raw da base limpa no GitHub.

url = "https://raw.githubusercontent.com/sidnei-almeida/mba-python-data-labs/refs/heads/master/data/telco_customer_churn_clean.csv"

# Carregando a base.
df = pd.read_csv(url)

# Visualizando as primeiras linhas.
df.head()

In [ ]:
# Teste do carregamento

assert df.shape[0] > 0, "A base precisa ter linhas."
assert "Churn" in df.columns, "A coluna Churn precisa existir."
assert "TotalCharges" in df.columns, "A coluna TotalCharges precisa existir."

print("Base carregada com sucesso!")
print("Linhas e colunas:", df.shape)

## Revisão rápida

Antes de modelar, confira rapidamente:

- tamanho da base;
- tipos das colunas;
- distribuição da variável alvo.

Mesmo em modelagem, sempre fazemos uma revisão inicial.

In [ ]:
# 2. Confira o tamanho da base.

df.shape

In [ ]:
# 3. Confira os tipos das colunas.

df.info()

In [ ]:
# 4. Confira a distribuição da variável alvo.

df["Churn"].value_counts()

In [ ]:
# Percentual da variável alvo.

df["Churn"].value_counts(normalize=True) * 100

# Parte 3 — Separar alvo e variáveis explicativas

Em Machine Learning supervisionado, geralmente separamos:

```text
X = variáveis explicativas
y = variável alvo
```

Neste projeto:

```text
y = Churn
```

O `X` será o restante das colunas que ajudam a prever o churn.

## Atenção

A coluna `customerID` é apenas um identificador do cliente.  
Ela não deve entrar no modelo.

Também vamos remover `Churn` de `X`, porque ela é justamente o que queremos prever.

## 5. Criar variável alvo `y`

Transforme `Churn` em números:

- `No` vira `0`;
- `Yes` vira `1`.

Salve o resultado em `y`.

Dica:

```python
y = df["Churn"].map({"No": 0, "Yes": 1})
```

In [ ]:
# 5. Crie a variável alvo y.

y = None

In [ ]:
# Teste do y

assert y is not None, "Crie a variável y."
assert set(y.unique()) == {0, 1}, "y deve conter apenas 0 e 1."
assert len(y) == len(df), "y deve ter o mesmo número de linhas que df."

print("Variável alvo criada corretamente!")
y.value_counts()

## 6. Criar matriz de variáveis `X`

Crie `X` removendo as colunas:

- `customerID`;
- `Churn`.

Dica:

```python
X = df.drop(["customerID", "Churn"], axis=1)
```

In [ ]:
# 6. Crie X removendo customerID e Churn.

X = None

In [ ]:
# Teste do X

assert X is not None, "Crie a variável X."
assert "Churn" not in X.columns, "Churn não pode estar em X."
assert "customerID" not in X.columns, "customerID não deve entrar no modelo."
assert len(X) == len(df), "X deve ter o mesmo número de linhas que df."

print("X criado corretamente!")
X.head()

# Parte 4 — Transformar categorias em números

Modelos do `scikit-learn` geralmente não entendem texto diretamente.

Exemplo:

```text
Contract = Month-to-month
PaymentMethod = Electronic check
InternetService = Fiber optic
```

Precisamos transformar essas categorias em colunas numéricas.

Neste primeiro modelo, vamos usar:

```python
pd.get_dummies()
```

Esse comando cria colunas 0/1 para categorias.

## 7. Aplicar `get_dummies`

Transforme `X` em uma versão numérica chamada `X_encoded`.

Use:

```python
X_encoded = pd.get_dummies(X, drop_first=True)
```

O parâmetro `drop_first=True` remove uma categoria de referência e evita colunas redundantes.

In [ ]:
# 7. Transforme as variáveis categóricas em números.

X_encoded = None

In [ ]:
# Teste do X_encoded

assert X_encoded is not None, "Crie X_encoded."
assert X_encoded.shape[0] == X.shape[0], "X_encoded deve ter o mesmo número de linhas que X."
assert X_encoded.select_dtypes(include="object").shape[1] == 0, "X_encoded não pode ter colunas de texto."

print("Encoding feito corretamente!")
print("Formato antes:", X.shape)
print("Formato depois:", X_encoded.shape)
X_encoded.head()

# Parte 5 — Separar treino e teste

Agora precisamos separar os dados em duas partes:

- **treino:** usado para o modelo aprender;
- **teste:** usado para avaliar se o modelo aprendeu algo útil.

Vamos usar:

```python
train_test_split()
```

Parâmetros importantes:

- `test_size=0.2`: 20% dos dados vão para teste;
- `random_state=42`: garante reprodutibilidade;
- `stratify=y`: mantém proporção parecida de churn no treino e no teste.

## 8. Criar treino e teste

Complete a célula usando `train_test_split`.

In [ ]:
# 8. Separe dados de treino e teste.

X_train, X_test, y_train, y_test = None

In [ ]:
# Teste do train/test split

assert X_train.shape[0] > X_test.shape[0], "O treino deve ter mais linhas que o teste."
assert X_train.shape[1] == X_test.shape[1], "Treino e teste devem ter a mesma quantidade de colunas."
assert len(y_train) == X_train.shape[0], "y_train deve ter o mesmo número de linhas que X_train."
assert len(y_test) == X_test.shape[0], "y_test deve ter o mesmo número de linhas que X_test."

print("Separação treino/teste correta!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# Parte 6 — Treinar primeiro modelo

Vamos usar uma **Árvore de Decisão**.

Para evitar que a árvore fique muito complexa, vamos limitar a profundidade:

```python
max_depth=4
```

Isso ajuda o modelo a ficar mais simples e mais fácil de interpretar.

## 9. Criar e treinar o modelo

Crie o modelo na variável `modelo`.

Depois treine com:

```python
modelo.fit(X_train, y_train)
```

In [ ]:
# 9. Crie e treine o modelo.

modelo = None

# Depois de criar o modelo, treine usando .fit()

# escreva seu código abaixo

In [ ]:
# Teste do modelo treinado

assert modelo is not None, "Crie o modelo."
assert hasattr(modelo, "predict"), "O modelo precisa ter o método predict."
assert hasattr(modelo, "classes_"), "Parece que o modelo ainda não foi treinado com .fit()."

print("Modelo treinado com sucesso!")

# Parte 7 — Fazer previsões

Depois que o modelo foi treinado, usamos `.predict()` para prever as classes dos dados de teste.

Salve as previsões na variável:

```python
y_pred
```

In [ ]:
# 10. Faça previsões no conjunto de teste.

y_pred = None

In [ ]:
# Teste das previsões

assert y_pred is not None, "Crie y_pred com modelo.predict(X_test)."
assert len(y_pred) == len(y_test), "y_pred deve ter o mesmo tamanho que y_test."
assert set(y_pred).issubset({0, 1}), "As previsões devem ser 0 ou 1."

print("Previsões criadas corretamente!")

# Parte 8 — Avaliar o modelo

Agora vamos avaliar o modelo.

A primeira métrica será a **acurácia**.

Acurácia responde:

> De todas as previsões feitas, qual percentual o modelo acertou?

Exemplo:

```text
accuracy = 0.78
```

significa que o modelo acertou aproximadamente 78% dos casos.

**Cuidado:** acurácia não conta a história toda, principalmente quando as classes são desbalanceadas.

## 11. Calcular acurácia

Use:

```python
accuracy_score(y_test, y_pred)
```

Salve em:

```python
acuracia
```

In [ ]:
# 11. Calcule a acurácia.

acuracia = None

In [ ]:
# Teste da acurácia

assert acuracia is not None, "Crie a variável acuracia."
assert 0 <= acuracia <= 1, "A acurácia deve estar entre 0 e 1."

print("Acurácia:", round(acuracia, 4))

## 12. Matriz de confusão

A matriz de confusão mostra os acertos e erros do modelo por classe.

Ela ajuda a responder:

- quantos clientes que não cancelaram foram previstos corretamente?
- quantos clientes que cancelaram foram previstos corretamente?
- onde o modelo está errando?

Use:

```python
confusion_matrix(y_test, y_pred)
```

Salve em:

```python
matriz_confusao
```

In [ ]:
# 12. Calcule a matriz de confusão.

matriz_confusao = None

In [ ]:
# Teste da matriz de confusão

assert matriz_confusao is not None, "Crie matriz_confusao."
assert matriz_confusao.shape == (2, 2), "Para classificação binária, a matriz deve ser 2x2."

print("Matriz de confusão:")
print(matriz_confusao)

## 13. Visualizar matriz de confusão

Vamos criar um gráfico simples da matriz de confusão.

Aqui o código está pronto, porque o foco não é decorar essa parte agora.

In [ ]:
plt.figure(figsize=(5, 4))

plt.imshow(matriz_confusao)

plt.title("Matriz de Confusão")
plt.xlabel("Previsto")
plt.ylabel("Real")

plt.xticks([0, 1], ["No", "Yes"])
plt.yticks([0, 1], ["No", "Yes"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, matriz_confusao[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

## 14. Relatório de classificação

O `classification_report` mostra várias métricas:

- **precision**
- **recall**
- **f1-score**
- **support**

Neste primeiro contato, foque principalmente em comparar as classes `0` e `1`.

A classe `1` representa clientes que cancelaram.

In [ ]:
# 14. Gere o classification report.

print(classification_report(y_test, y_pred, target_names=["No", "Yes"]))

# Parte 9 — Comparar com um baseline simples

Um baseline é uma comparação simples.

Por exemplo:

> E se um modelo chutasse sempre a classe mais comum?

Como a maioria dos clientes não cancela, um modelo muito bobo poderia prever sempre `No`.

Se nosso modelo não for muito melhor que isso, temos um problema.

In [ ]:
# Baseline: classe mais comum no treino.

classe_mais_comum = y_train.value_counts().idxmax()

# Criando uma previsão boba: todo mundo recebe a classe mais comum.
y_pred_baseline = [classe_mais_comum] * len(y_test)

# Acurácia do baseline.
acuracia_baseline = accuracy_score(y_test, y_pred_baseline)

print("Classe mais comum:", classe_mais_comum)
print("Acurácia baseline:", round(acuracia_baseline, 4))
print("Acurácia modelo:", round(acuracia, 4))

## 15. Interpretação do baseline

Responda em Markdown:

1. O modelo foi melhor que o baseline?
2. A diferença foi grande ou pequena?
3. A acurácia sozinha parece suficiente para avaliar o modelo?
4. O modelo parece bom para encontrar clientes que cancelam?

## Minha interpretação

Escreva aqui:

1. 
2. 
3. 
4.

# Parte 10 — Importância das variáveis

Árvores de decisão conseguem indicar quais variáveis foram mais importantes para o modelo.

Isso não significa causalidade, mas ajuda a entender quais colunas o modelo usou mais.

In [ ]:
# Criando uma tabela com importância das variáveis.

importancias = pd.DataFrame({
    "feature": X_encoded.columns,
    "importance": modelo.feature_importances_
})

importancias = importancias.sort_values("importance", ascending=False)

importancias.head(10)

In [ ]:
# Gráfico das 10 variáveis mais importantes.

top_10_importancias = importancias.head(10)

plt.figure(figsize=(10, 5))

plt.barh(top_10_importancias["feature"], top_10_importancias["importance"])

plt.title("Top 10 Variáveis Mais Importantes")
plt.xlabel("Importância")
plt.ylabel("Variável")

plt.gca().invert_yaxis()

plt.show()

## 16. Interpretação das variáveis importantes

Responda em Markdown:

1. Quais variáveis apareceram como mais importantes?
2. Elas fazem sentido com a análise do Lab 03?
3. Alguma variável surpreendeu?
4. Que hipóteses você levantaria a partir disso?

## Minha interpretação das variáveis

Escreva aqui:

1. 
2. 
3. 
4.

# Parte 11 — Experimento rápido

Agora vamos fazer um pequeno experimento.

Treine uma nova árvore com profundidade diferente:

```python
max_depth=6
```

Compare com o modelo anterior.

Salve o novo modelo em:

```python
modelo_2
```

Salve as previsões em:

```python
y_pred_2
```

Salve a acurácia em:

```python
acuracia_2
```

In [ ]:
# 17. Treine uma segunda árvore com max_depth=6.

modelo_2 = None
y_pred_2 = None
acuracia_2 = None

In [ ]:
# Teste do experimento

assert modelo_2 is not None, "Crie modelo_2."
assert y_pred_2 is not None, "Crie y_pred_2."
assert acuracia_2 is not None, "Crie acuracia_2."
assert 0 <= acuracia_2 <= 1, "A acurácia deve estar entre 0 e 1."

print("Acurácia modelo 1:", round(acuracia, 4))
print("Acurácia modelo 2:", round(acuracia_2, 4))

## 18. Conclusão do experimento

Responda:

1. A árvore com `max_depth=6` melhorou a acurácia?
2. Melhorar acurácia sempre significa melhorar o modelo?
3. O que você compararia além da acurácia?

## Minha conclusão do experimento

Escreva aqui:

1. 
2. 
3.

# Parte 12 — Salvando o modelo treinado

Até aqui, o modelo existe apenas na memória do notebook.

Em um projeto real, depois de treinar um modelo, geralmente queremos salvar esse modelo em um arquivo para usar depois.

Para isso, vamos usar a biblioteca `joblib`.

Vamos salvar:

- o modelo treinado;
- a lista de colunas usadas no treino.

Por que salvar as colunas?

Porque quando formos fazer previsão em novos dados, os dados novos precisam ter **as mesmas colunas** que o modelo viu durante o treino.

## 19. Salvar o modelo em `.joblib`

Salve o modelo treinado em:

```text
modelo_churn_decision_tree.joblib
```

Dica:

```python
joblib.dump(modelo, "modelo_churn_decision_tree.joblib")
```

In [ ]:
# 19. Salve o modelo treinado em um arquivo joblib.

# escreva seu código abaixo

In [ ]:
# Teste do arquivo do modelo

from pathlib import Path

assert Path("modelo_churn_decision_tree.joblib").exists(), "O arquivo do modelo não foi encontrado."

print("Modelo salvo com sucesso!")

## 20. Salvar as colunas usadas no treino

Agora salve a lista de colunas de `X_encoded`.

Use o nome:

```text
colunas_modelo_churn.joblib
```

Dica:

```python
joblib.dump(list(X_encoded.columns), "colunas_modelo_churn.joblib")
```

In [ ]:
# 20. Salve a lista de colunas usadas no treino.

# escreva seu código abaixo

In [ ]:
# Teste do arquivo de colunas

assert Path("colunas_modelo_churn.joblib").exists(), "O arquivo com as colunas do modelo não foi encontrado."

print("Colunas do modelo salvas com sucesso!")

# Parte 13 — Carregando o modelo novamente

Agora vamos simular uma situação real:

> O notebook foi fechado, mas queremos carregar o modelo salvo e usar para prever novamente.

Vamos carregar:

- o modelo salvo;
- a lista de colunas usadas no treino.

In [ ]:
# Carregando modelo e colunas salvas.

modelo_carregado = joblib.load("modelo_churn_decision_tree.joblib")
colunas_carregadas = joblib.load("colunas_modelo_churn.joblib")

print("Modelo carregado:", modelo_carregado)
print("Quantidade de colunas carregadas:", len(colunas_carregadas))

In [ ]:
# Teste do modelo carregado

assert hasattr(modelo_carregado, "predict"), "O modelo carregado precisa ter predict."
assert isinstance(colunas_carregadas, list), "As colunas carregadas precisam estar em uma lista."
assert colunas_carregadas == list(X_encoded.columns), "As colunas carregadas precisam bater com X_encoded."

print("Modelo e colunas carregados corretamente!")

# Parte 14 — Fazendo previsão com o modelo carregado

Vamos testar se o modelo carregado consegue fazer previsões.

Primeiro, vamos usar algumas linhas do próprio conjunto de teste.

In [ ]:
# Fazendo previsões com o modelo carregado usando X_test.

y_pred_carregado = modelo_carregado.predict(X_test)

# Comparando com as previsões do modelo original.
(y_pred_carregado == y_pred).mean()

In [ ]:
# Teste das previsões do modelo carregado

assert len(y_pred_carregado) == len(y_test), "As previsões precisam ter o mesmo tamanho de y_test."
assert (y_pred_carregado == y_pred).all(), "O modelo carregado deveria gerar as mesmas previsões do modelo original."

print("Modelo carregado gerou as mesmas previsões!")

# Parte 15 — Simulando novos clientes

Agora vamos simular novos dados.

Em um cenário real, você receberia novos clientes e precisaria prever se eles têm risco de churn.

Para simplificar, vamos pegar algumas linhas da base original e fingir que são novos clientes.

Depois, precisamos aplicar **o mesmo pré-processamento**:

1. remover `customerID` e `Churn`, se existirem;
2. aplicar `pd.get_dummies()`;
3. alinhar as colunas com as colunas usadas no treino;
4. chamar `.predict()`.

In [ ]:
# Selecionando 5 clientes como se fossem novos dados.

novos_clientes = df.sample(5, random_state=42)

novos_clientes

## 21. Preparar novos clientes para previsão

Crie `novos_clientes_X` removendo as colunas:

- `customerID`;
- `Churn`.

Depois aplique `pd.get_dummies()` e salve em `novos_clientes_encoded`.

Dica:

```python
novos_clientes_X = novos_clientes.drop(["customerID", "Churn"], axis=1)
novos_clientes_encoded = pd.get_dummies(novos_clientes_X, drop_first=True)
```

In [ ]:
# 21. Prepare os novos clientes.

novos_clientes_X = None
novos_clientes_encoded = None

In [ ]:
# Teste da preparação dos novos clientes

assert novos_clientes_X is not None, "Crie novos_clientes_X."
assert novos_clientes_encoded is not None, "Crie novos_clientes_encoded."
assert "customerID" not in novos_clientes_X.columns, "customerID não deve estar em novos_clientes_X."
assert "Churn" not in novos_clientes_X.columns, "Churn não deve estar em novos_clientes_X."
assert novos_clientes_encoded.select_dtypes(include="object").shape[1] == 0, "novos_clientes_encoded não pode ter texto."

print("Novos clientes preparados parcialmente!")

## 22. Alinhar colunas dos novos clientes

Aqui está uma parte muito importante.

Quando usamos `pd.get_dummies()` em poucos clientes, algumas categorias podem não aparecer.  
Então o DataFrame novo pode ficar com menos colunas que o treino.

Para resolver isso, usamos:

```python
reindex(columns=colunas_carregadas, fill_value=0)
```

Isso garante que os novos dados tenham exatamente as mesmas colunas do treino.

In [ ]:
# 22. Alinhe os novos clientes com as colunas do modelo.

novos_clientes_encoded = None

In [ ]:
# Teste do alinhamento

assert list(novos_clientes_encoded.columns) == colunas_carregadas, "As colunas precisam estar iguais às colunas do treino."
assert novos_clientes_encoded.shape[1] == len(colunas_carregadas), "A quantidade de colunas precisa bater."

print("Colunas alinhadas corretamente!")
print("Formato dos novos dados:", novos_clientes_encoded.shape)

## 23. Prever churn dos novos clientes

Use o modelo carregado para prever os novos clientes.

Salve o resultado em:

```python
previsoes_novos_clientes
```

In [ ]:
# 23. Faça previsões para os novos clientes.

previsoes_novos_clientes = None

In [ ]:
# Teste das previsões dos novos clientes

assert previsoes_novos_clientes is not None, "Crie previsoes_novos_clientes."
assert len(previsoes_novos_clientes) == len(novos_clientes), "Deve existir uma previsão para cada novo cliente."
assert set(previsoes_novos_clientes).issubset({0, 1}), "As previsões devem ser 0 ou 1."

print("Previsões geradas com sucesso!")
previsoes_novos_clientes

## 24. Criar uma tabela com as previsões

Agora vamos criar uma tabela final mais legível.

A previsão `0` significa `No`.  
A previsão `1` significa `Yes`.

In [ ]:
# Criando tabela de resultados.

resultado_previsoes = novos_clientes[["customerID", "Contract", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].copy()

resultado_previsoes["Predicted Churn"] = previsoes_novos_clientes
resultado_previsoes["Predicted Churn Label"] = resultado_previsoes["Predicted Churn"].map({0: "No", 1: "Yes"})

resultado_previsoes

## 25. Reflexão sobre uso real do modelo

Responda em Markdown:

1. Por que precisamos salvar o modelo?
2. Por que precisamos salvar também as colunas usadas no treino?
3. O que poderia dar errado se os novos dados tivessem colunas diferentes?
4. Esse modelo já estaria pronto para produção? Por quê?

## Minha reflexão

Escreva aqui:

1. 
2. 
3. 
4.

# Desafio final — Resumo do projeto

Escreva um pequeno resumo do projeto em Markdown.

Inclua:

- objetivo do modelo;
- dataset usado;
- principais etapas;
- acurácia do modelo;
- comparação com baseline;
- principais variáveis importantes;
- uma conclusão de negócio.

Esse texto pode virar parte do README do projeto no GitHub.

## Resumo final

Escreva aqui seu resumo:

# Fechamento

Neste lab, você praticou o fluxo básico de classificação:

1. carregar dados limpos;
2. separar `X` e `y`;
3. transformar categorias em números;
4. separar treino e teste;
5. treinar um modelo;
6. fazer previsões;
7. avaliar acurácia;
8. analisar matriz de confusão;
9. comparar com baseline;
10. interpretar importância das variáveis;
11. salvar modelo com `joblib`;
12. carregar modelo salvo;
13. fazer previsões em novos dados.

Este já é o primeiro passo real para um projeto de portfólio:

```text
Customer Churn Prediction with Scikit-learn
```

Em um próximo lab/projeto, podemos melhorar este modelo usando:

- `RandomForestClassifier`;
- `LogisticRegression`;
- normalização;
- pipelines;
- validação cruzada;
- tratamento de desbalanceamento;
- métricas como recall e F1-score.